# TRIBE v2 Surface Decoder UI

This UI-only Colab notebook launches Gradio. Setup, dependency install, TRIBE inference, cache reuse, surface decoding, and visualization are all triggered from the interface button.


In [ ]:
# -*- coding: utf-8 -*-
import base64
import os
import pickle
import shutil
import subprocess
import sys
import traceback
from pathlib import Path


def parse_version_prefix(raw: str) -> tuple[int, int]:
    """Return major/minor from a package version string."""

    parts = []
    for token in raw.replace("-", ".").split("."):
        if token.isdigit():
            parts.append(int(token))
        else:
            digits = "".join(char for char in token if char.isdigit())
            if digits:
                parts.append(int(digits))
        if len(parts) >= 2:
            break
    while len(parts) < 2:
        parts.append(0)
    return parts[0], parts[1]


def ensure_numpy_compatibility() -> None:
    """Pin NumPy before Gradio/TRIBE dependencies import it in Colab."""

    from importlib.metadata import PackageNotFoundError, version

    try:
        numpy_version = version("numpy")
    except PackageNotFoundError:
        numpy_version = ""

    needs_pin = not numpy_version or parse_version_prefix(numpy_version) >= (2, 1)
    if not needs_pin:
        print(f"NumPy compatible with TRIBE plotting: {numpy_version}")
        return

    numpy_already_loaded = "numpy" in sys.modules
    print(f"Installing NumPy <2.1 for TRIBE plotting compatibility. Current: {numpy_version or 'not installed'}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy<2.1"],
        check=True,
    )
    if numpy_already_loaded:
        raise RuntimeError(
            "NumPy was already imported in this kernel before it was pinned. "
            "Restart the Colab runtime once, then rerun this UI cell."
        )


ensure_numpy_compatibility()

try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "gradio>=4.44,<6", "numpy<2.1"], check=True)
    import gradio as gr

PROJECT_DIR = Path("/content/neuromedia")
DEFAULT_ROOT_DIR = "/content/drive/MyDrive/neuromedia"
DEFAULT_INPUT_PATH = "/content/drive/MyDrive/neuromedia/input/4590080h.mp4"
TRIBE_B64 = "..."
SURFACE_DECODER_B64 = "..."
REPORT_B64 = "..."
REQUIREMENTS_B64 = "bnVtcHk+PTEuMjYuNCw8Mi4xCnBhbmRhcz49Mi4yLDwzCnNjaXB5Pj0xLjEzLDwxLjE1Cm5pYmFiZWw+PTUuMiw8NgpuaWxlYXJuPj0wLjExLDwwLjEzCnNjaWtpdC1sZWFybj49MS40LDwxLjgKZ3JhZGlvPj00LjQ0LDw2Cm1hdHBsb3RsaWI+PTMuOCw8My4xMQpzZWFib3JuPj0wLjEzLDwwLjE0CmNvbG9yY2V0Pj0zLDw0CnB5dmlzdGE+PTAuNDMsPDAuNDcKc2Npa2l0LWltYWdlPj0wLjIzLDwwLjI2CnB5YXJyb3c+PTE3LDwyMwptbmU+PTEuMTAsPDIKbW5lLWJpZHM+PTAuMTYsPDAuMTgKcHlwcmVwPj0wLjQuMyw8MC42CnRxZG0+PTQuNjUsPDUKZXhjYT49MC41LjIwLDwwLjYKcGFja2FnaW5nPj0yNCw8MjcKcG9sYXJzPj0xLDwyCnVqc29uPj01LDw2CnB5ZGFudGljPj0yLjUsPDMKdG9yY2g+PTIuNS4xLDwyLjcKdG9yY2h2aXNpb24+PTAuMjAsPDAuMjIKdG9yY2htZXRyaWNzPj0xLjEsPDIKeF90cmFuc2Zvcm1lcnM9PTEuMjcuMjAKZWlub3BzPj0wLjgsPDAuOQpweXlhbWw+PTYsPDcKbW92aWVweT49Mi4yLjEsPDMKaHVnZ2luZ2ZhY2VfaHViPj0wLjI0LDwxCmd0dHM+PTIuNSw8MwpsYW5nZGV0ZWN0Pj0xLjAuOSw8MgpzcGFjeT49My44LDw0CnNvdW5kZmlsZT49MC4xMyw8MC4xNApMZXZlbnNodGVpbj49MC4yNyw8MC4yOApqdWxpdXM+PTAuMi43LDwwLjMKdHJhbnNmb3JtZXJzPj00LjQ1LDw1CnBpbGxvdz49OS4yLDwxMgo="
TRIBE_NODEPS_REQUIREMENTS_B64 = "bmV1cmFsc2V0PT0wLjAuMgpuZXVyYWx0cmFpbj09MC4wLjIKdHJpYmV2MltwbG90dGluZ10gQCBnaXQraHR0cHM6Ly9naXRodWIuY29tL2ZhY2Vib29rcmVzZWFyY2gvdHJpYmV2Mi5naXQK"
SUPPORTED_EXTENSIONS = {".txt", ".wav", ".mp3", ".flac", ".ogg", ".mp4", ".avi", ".mkv", ".mov", ".webm"}


def write_project_files() -> None:
    """Write embedded project files into the Colab runtime."""

    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "tribe_nimare_interpreter.py").write_text(
        base64.b64decode(TRIBE_B64).decode("utf-8"), encoding="utf-8"
    )
    (PROJECT_DIR / "marketing_surface_decoder.py").write_text(
        base64.b64decode(SURFACE_DECODER_B64).decode("utf-8"), encoding="utf-8"
    )
    (PROJECT_DIR / "marketing_report.py").write_text(
        base64.b64decode(REPORT_B64).decode("utf-8"), encoding="utf-8"
    )
    (PROJECT_DIR / "requirements.txt").write_text(
        base64.b64decode(REQUIREMENTS_B64).decode("utf-8"), encoding="utf-8"
    )
    (PROJECT_DIR / "requirements-tribe-nodeps.txt").write_text(
        base64.b64decode(TRIBE_NODEPS_REQUIREMENTS_B64).decode("utf-8"), encoding="utf-8"
    )


def output_dir_for_input(input_path: Path, input_dir: Path, tribe_output_root: Path) -> Path:
    """Return stable TRIBE output directory for an input file."""

    try:
        relative = input_path.relative_to(input_dir).with_suffix("")
        return tribe_output_root.joinpath(*relative.parts)
    except ValueError:
        return tribe_output_root / input_path.stem


def run_command_stream(cmd: list[str], log_lines: list[str], env: dict[str, str] | None = None):
    """Run a command and append stdout/stderr lines to the UI log."""

    log_lines.append("$ " + " ".join(cmd))
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        log_lines.append(line.rstrip())
        yield
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


def is_active_mount(path: Path) -> bool:
    """Return True when path is an active OS mount point."""

    try:
        if os.path.ismount(path):
            return True
    except Exception:
        pass
    try:
        mounts = Path("/proc/mounts").read_text(encoding="utf-8").splitlines()
    except Exception:
        return False
    path_text = str(path)
    return any(len(line.split()) > 1 and line.split()[1] == path_text for line in mounts)


def maybe_mount_drive(enabled: bool, force_remount: bool, log_lines: list[str]) -> None:
    """Mount Google Drive when running inside Colab and the user requested it."""

    if not enabled:
        log_lines.append("Drive mount skipped by user.")
        return
    mount_point = Path("/content/drive")
    try:
        from google.colab import drive
    except Exception as exc:
        log_lines.append(f"Google Drive mount unavailable in this environment: {exc}")
        return

    if force_remount:
        log_lines.append("Force-remounting Google Drive before run.")
        try:
            drive.flush_and_unmount()
            log_lines.append("Existing Google Drive mount flushed/unmounted.")
        except Exception as exc:
            log_lines.append(f"Drive flush/unmount skipped or failed: {exc}")
        if is_active_mount(mount_point):
            raise RuntimeError(
                "Refusing to remove /content/drive because it is still an active mount. "
                "Restart the Colab runtime and try again."
            )
        if mount_point.exists():
            shutil.rmtree(mount_point)
        mount_point.mkdir(parents=True, exist_ok=True)
        drive.mount(str(mount_point), force_remount=True)
        log_lines.append("Google Drive force-remounted.")
        return

    if Path("/content/drive/MyDrive").exists():
        log_lines.append("Google Drive already mounted.")
        return
    log_lines.append("Mounting Google Drive. Complete the auth prompt in the notebook output if it appears.")
    drive.mount(str(mount_point))


def describe_missing_input(input_path: Path, input_dir: Path) -> str:
    """Build a useful error message when an input path is not visible."""

    drive_root = Path("/content/drive/MyDrive")
    parent = input_path.parent
    search_dir = parent if parent.exists() else input_dir
    lines = [
        f"Input file not found: {input_path}",
        f"Google Drive mounted: {drive_root.exists()}",
        f"Requested parent exists: {parent.exists()}",
        "Colab paths are case-sensitive; check the exact file extension too.",
        "If the file is visible in Google Drive web UI but not here, force-remount Drive in Colab.",
        "Remount snippet: from google.colab import drive; drive.flush_and_unmount(); drive.mount('/content/drive', force_remount=True)",
    ]

    lines.append(f"Visible files in {search_dir}:")
    if search_dir.exists():
        visible_files = sorted(path for path in search_dir.iterdir() if path.is_file())
        if visible_files:
            lines.extend(f"- {path.name}" for path in visible_files[:50])
            if len(visible_files) > 50:
                lines.append(f"- ... {len(visible_files) - 50} more files")
        else:
            lines.append("- <no files>")
    else:
        lines.append("- <directory does not exist>")

    if input_dir.exists():
        similar_files = sorted(input_dir.rglob(f"{input_path.stem}.*"))
        if similar_files:
            lines.append("Files with the same name stem under project input:")
            lines.extend(f"- {path}" for path in similar_files[:20])
    return "\n".join(lines)


def read_scores(output_dir: Path):
    """Read aggregate and per-segment marketing score tables."""

    import pandas as pd

    scores_path = output_dir / "marketing_scores.csv"
    terms_path = output_dir / "decoded_terms.csv"
    report_path = output_dir / "report.json"
    missing = [str(path) for path in (scores_path, terms_path, report_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError("Decode finished without expected output files: " + ", ".join(missing))

    scores = pd.read_csv(scores_path)
    if scores.empty:
        raise RuntimeError(f"Marketing scores file is empty: {scores_path}")

    aggregate = scores[scores["time_index"].astype(str) == "aggregate"].copy()
    aggregate = aggregate.sort_values(["map_id", "score_0_100"], ascending=[True, False])
    if aggregate.empty:
        raise RuntimeError(f"No aggregate rows found in {scores_path}")

    segment_scores = scores[
        (scores["map_id"] == "tribe_predictions_fsaverage5")
        & (scores["time_index"].astype(str) != "aggregate")
    ].copy()
    if segment_scores.empty:
        return aggregate, pd.DataFrame()

    segment_scores["time_index_int"] = segment_scores["time_index"].astype(int)
    segment_scores = segment_scores.sort_values(
        ["time_index_int", "score_0_100"], ascending=[True, False]
    )
    rows = []
    for time_index, group_df in segment_scores.groupby("time_index_int", sort=True):
        for rank, (_, row) in enumerate(group_df.head(3).iterrows(), start=1):
            rows.append(
                {
                    "segment": int(time_index),
                    "rank": rank,
                    "group": row["group"],
                    "score_0_100": float(row["score_0_100"]),
                    "mean_r": float(row["mean_r"]),
                    "n_features": int(row["n_features"]),
                }
            )
    return aggregate, pd.DataFrame(rows)


def load_plot_segments(tribe_dir: Path, n_timesteps: int, log_lines: list[str]):
    """Load full TRIBE segment objects for the official demo visualization."""

    pkl_path = tribe_dir / "tribe_segments.pkl"
    if pkl_path.is_file():
        try:
            with pkl_path.open("rb") as handle:
                segments = list(pickle.load(handle))
            if segments:
                log_lines.append(f"Loaded full TRIBE segments for official plot: {pkl_path}")
                return segments[:n_timesteps], True
            log_lines.append(f"Full TRIBE segments file is empty: {pkl_path}")
        except Exception as exc:
            log_lines.append(f"Could not load full TRIBE segments from {pkl_path}: {type(exc).__name__}: {exc}")

    segments_path = tribe_dir / "tribe_segments.tsv"
    if segments_path.is_file():
        log_lines.append(
            "Only tribe_segments.tsv is available; video/audio/text rows need tribe_segments.pkl. "
            "Rerun TRIBE once with official stimuli plot enabled to create it."
        )
    else:
        log_lines.append(f"TRIBE segments not found: {segments_path}")
    return None, False


def make_tribe_plot(tribe_dir: Path, n_timesteps: int, try_show_stimuli: bool, log_lines: list[str]):
    """Create official TRIBE timestep visualization from cached predictions."""

    import numpy as np

    preds_path = tribe_dir / "tribe_predictions_fsaverage5.npy"
    if not preds_path.is_file():
        log_lines.append(f"Visualization skipped: predictions not found: {preds_path}")
        return None

    preds = np.load(preds_path).astype(np.float32)
    n_timesteps = max(1, min(int(n_timesteps), preds.shape[0]))

    try:
        from tribev2.plotting import PlotBrain

        plotter = PlotBrain(mesh="fsaverage5")
        segments, has_full_segments = load_plot_segments(tribe_dir, n_timesteps, log_lines)
        if try_show_stimuli and has_full_segments:
            try:
                return plotter.plot_timesteps(
                    preds[:n_timesteps],
                    segments=segments[:n_timesteps],
                    cmap="fire",
                    norm_percentile=99,
                    vmin=0.6,
                    alpha_cmap=(0, 0.2),
                    show_stimuli=True,
                )
            except Exception as exc:
                log_lines.append(
                    "Official TRIBE plot with stimuli failed; retrying brain-only plot: "
                    f"{type(exc).__name__}: {exc}"
                )
        elif try_show_stimuli:
            log_lines.append("Official stimuli plot skipped because full TRIBE segments are not available.")

        return plotter.plot_timesteps(
            preds[:n_timesteps],
            segments=None,
            cmap="fire",
            norm_percentile=99,
            vmin=0.6,
            alpha_cmap=(0, 0.2),
            show_stimuli=False,
        )
    except Exception as exc:
        log_lines.append(f"TRIBE plotter unavailable; using fallback heatmap: {type(exc).__name__}: {exc}")
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(10, 3.5))
        im = ax.imshow(preds[:n_timesteps], aspect="auto", interpolation="nearest", cmap="inferno")
        ax.set_title("TRIBE fsaverage5 predictions, fallback heatmap")
        ax.set_xlabel("fsaverage5 vertex")
        ax.set_ylabel("segment")
        fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
        return fig



def persist_run_log(log_path: Path | None, log_lines: list[str]) -> None:
    """Persist the latest UI log to Drive/local disk for debugging."""

    if log_path is None:
        return
    try:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("\n".join(log_lines), encoding="utf-8")
    except Exception:
        pass


def describe_output_dir(output_dir: Path) -> str:
    """Return a compact listing of output files for debug logs."""

    if not output_dir.exists():
        return f"Output dir does not exist: {output_dir}"
    rows = []
    for path in sorted(output_dir.glob("*")):
        if path.is_file():
            rows.append(f"{path.name}: {path.stat().st_size} bytes")
        else:
            rows.append(f"{path.name}/")
    return "\n".join(rows) if rows else f"Output dir is empty: {output_dir}"


def analyze_video(
    video_path: str,
    hf_token: str,
    root_dir: str,
    install_requirements: bool,
    mount_drive: bool,
    force_remount_drive: bool,
    force_rerun_tribe: bool,
    n_timesteps: int,
    show_stimuli: bool,
):
    """Gradio generator that performs setup, TRIBE, decode and visualization."""

    import pandas as pd

    log_lines: list[str] = []
    empty_df = pd.DataFrame()
    log_path: Path | None = None

    def snapshot(status: str, fig=None, aggregate=None, segments=None, paths_text: str = "", report_file=None):
        persist_run_log(log_path, log_lines)
        return (
            status,
            "\n".join(log_lines[-320:]),
            paths_text,
            fig,
            aggregate if aggregate is not None else empty_df,
            segments if segments is not None else empty_df,
            report_file,
        )

    try:
        os.environ.setdefault("PYTHONUTF8", "1")
        os.environ.setdefault("PYTHONUNBUFFERED", "1")
        write_project_files()
        log_lines.append(f"Project files written to {PROJECT_DIR}")
        yield snapshot("Project files ready")

        if mount_drive:
            maybe_mount_drive(True, force_remount_drive, log_lines)
            yield snapshot("Drive checked")

        token = (hf_token or "").strip()
        if token:
            os.environ["HF_TOKEN"] = token
            os.environ["HUGGING_FACE_HUB_TOKEN"] = token
            log_lines.append("HF token loaded from UI field.")
        else:
            log_lines.append("HF token field is empty. Cached/public models may still work.")

        env = os.environ.copy()
        env["PYTHONUTF8"] = "1"
        env["PYTHONUNBUFFERED"] = "1"

        if install_requirements:
            log_lines.append(
                "Installing/updating Python requirements. This can take several minutes on a fresh Colab runtime. "
                "TRIBE packages are installed in a second --no-deps step to avoid their invalid NumPy pins."
            )
            install_cmd = [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")]
            for _ in run_command_stream(install_cmd, log_lines, env=env):
                yield snapshot("Installing requirements")
            tribe_install_cmd = [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-deps",
                "-r",
                str(PROJECT_DIR / "requirements-tribe-nodeps.txt"),
            ]
            for _ in run_command_stream(tribe_install_cmd, log_lines, env=env):
                yield snapshot("Installing TRIBE packages")
        else:
            log_lines.append("Requirements install skipped by user.")

        root = Path(root_dir).expanduser()
        log_path = root / "logs" / "gradio_last_run.log"
        log_lines.append(f"Debug log will be saved to {log_path}")
        input_dir = root / "input"
        output_root = root / "outputs" / "surface_decoder"
        tribe_output_root = root / "outputs" / "tribe_fsaverage5"
        tribe_cache_dir = root / "cache" / "tribev2"
        references_dir = root / "cache" / "surface_references" / "marketing_v1"
        neurosynth_cache_dir = root / "cache" / "neurosynth_precomputed"
        for path in (input_dir, output_root, tribe_output_root, tribe_cache_dir, references_dir, neurosynth_cache_dir):
            path.mkdir(parents=True, exist_ok=True)

        input_path = Path(video_path).expanduser()
        if not input_path.is_file():
            raise FileNotFoundError(describe_missing_input(input_path, input_dir))
        if input_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            raise ValueError(f"Unsupported input extension: {input_path.suffix}")

        tribe_dir = output_dir_for_input(input_path, input_dir, tribe_output_root)
        surface_dir = output_root.joinpath(*tribe_dir.relative_to(tribe_output_root).parts)
        report_html = surface_dir / "marketing_report.html"
        tribe_dir.mkdir(parents=True, exist_ok=True)
        surface_dir.mkdir(parents=True, exist_ok=True)
        paths_text = (
            f"Input: {input_path}\n"
            f"TRIBE output: {tribe_dir}\n"
            f"Decode output: {surface_dir}\n"
            f"Scores: {surface_dir / 'marketing_scores.csv'}\n"
            f"HTML report: {report_html}"
        )
        yield snapshot("Paths ready", paths_text=paths_text)

        reference_maps = references_dir / "reference_maps.npy"
        reference_metadata = references_dir / "reference_metadata.json"
        if not reference_maps.is_file() or not reference_metadata.is_file():
            log_lines.append("Reference maps not found. Building lightweight Neurosynth references...")
            build_cmd = [
                sys.executable,
                str(PROJECT_DIR / "marketing_surface_decoder.py"),
                "build-references",
                "--references-dir",
                str(references_dir),
                "--reference-source",
                "neurosynth_precomputed",
                "--neurosynth-cache-dir",
                str(neurosynth_cache_dir),
                "--n-cores",
                "1",
            ]
            for _ in run_command_stream(build_cmd, log_lines, env=env):
                yield snapshot("Building reference maps", paths_text=paths_text)
        else:
            log_lines.append(f"Using cached reference maps: {references_dir}")
            yield snapshot("Reference maps cached", paths_text=paths_text)

        prediction_file = tribe_dir / "tribe_predictions_fsaverage5.npy"
        activity_file = tribe_dir / "tribe_activity_fsaverage5.npy"
        segments_pickle_file = tribe_dir / "tribe_segments.pkl"
        has_tribe_cache = prediction_file.is_file() and activity_file.is_file()
        needs_full_segments = bool(show_stimuli)
        can_reuse_tribe = (
            has_tribe_cache
            and not force_rerun_tribe
            and (not needs_full_segments or segments_pickle_file.is_file())
        )
        if can_reuse_tribe:
            log_lines.append(f"Using cached TRIBE outputs: {tribe_dir}")
            yield snapshot("Using cached TRIBE outputs", paths_text=paths_text)
        else:
            if has_tribe_cache and not force_rerun_tribe and needs_full_segments and not segments_pickle_file.is_file():
                log_lines.append(
                    "Cached .npy outputs exist, but full tribe_segments.pkl is missing. "
                    "Rerunning TRIBE once so the official video/audio/text visualization can be built. "
                    "Disable the stimuli checkbox to reuse old cache without rerun."
                )
            tribe_cmd = [
                sys.executable,
                str(PROJECT_DIR / "tribe_nimare_interpreter.py"),
                str(input_path),
                "--output-dir",
                str(tribe_dir),
                "--cache-dir",
                str(tribe_cache_dir),
                "--device",
                "cuda",
                "--tribe-only",
            ]
            log_lines.append("Running TRIBE v2...")
            for _ in run_command_stream(tribe_cmd, log_lines, env=env):
                yield snapshot("Running TRIBE v2", paths_text=paths_text)

        decode_inputs = []
        if prediction_file.is_file():
            decode_inputs.append(prediction_file)
        if activity_file.is_file():
            decode_inputs.append(activity_file)
        if not decode_inputs:
            raise FileNotFoundError(f"No TRIBE .npy outputs found in {tribe_dir}")

        decode_cmd = [
            sys.executable,
            str(PROJECT_DIR / "marketing_surface_decoder.py"),
            "decode",
            *[str(path) for path in decode_inputs],
            "--references-dir",
            str(references_dir),
            "--output-dir",
            str(surface_dir),
            "--hemisphere-order",
            "left_then_right",
        ]
        log_lines.append("Running surface decoder...")
        for _ in run_command_stream(decode_cmd, log_lines, env=env):
            yield snapshot("Decoding", paths_text=paths_text)

        log_lines.append("Decode output files:")
        log_lines.append(describe_output_dir(surface_dir))
        aggregate_df, segment_top_df = read_scores(surface_dir)
        log_lines.append(f"Loaded aggregate rows: {len(aggregate_df)}; segment top rows: {len(segment_top_df)}")

        report_cmd = [
            sys.executable,
            str(PROJECT_DIR / "marketing_report.py"),
            "--tribe-dir",
            str(tribe_dir),
            "--surface-dir",
            str(surface_dir),
            "--output-html",
            str(report_html),
            "--input-media",
            str(input_path),
            "--title",
            f"TRIBE v2 marketing report: {input_path.name}",
        ]
        log_lines.append("Building downloadable HTML report with all TRIBE timesteps...")
        for _ in run_command_stream(report_cmd, log_lines, env=env):
            yield snapshot("Building report", paths_text=paths_text)
        if not report_html.is_file():
            raise FileNotFoundError(f"HTML report was not created: {report_html}")

        fig = make_tribe_plot(tribe_dir, n_timesteps, show_stimuli, log_lines)
        yield snapshot(
            "Done",
            fig=fig,
            aggregate=aggregate_df,
            segments=segment_top_df,
            paths_text=paths_text,
            report_file=str(report_html),
        )
    except Exception:
        log_lines.append(traceback.format_exc())
        persist_run_log(log_path, log_lines)
        yield snapshot("Failed")


with gr.Blocks(title="TRIBE v2 Surface Decoder") as demo:
    gr.Markdown("# TRIBE v2 + Surface Marketing Decoder")
    gr.Markdown("Choose one input file. Nothing is processed until you click **Run analysis**. The downloadable HTML report includes all TRIBE timesteps, not only the preview limit.")

    with gr.Row():
        video_path_input = gr.Textbox(label="Input file path", value=DEFAULT_INPUT_PATH, lines=1)
        root_dir_input = gr.Textbox(label="Project root", value=DEFAULT_ROOT_DIR, lines=1)
    hf_token_input = gr.Textbox(label="Hugging Face token (optional, not saved)", type="password", lines=1)

    with gr.Row():
        install_requirements_input = gr.Checkbox(label="Install/update requirements before run", value=True)
        mount_drive_input = gr.Checkbox(label="Mount Google Drive if needed", value=True)
        force_remount_drive_input = gr.Checkbox(label="Force Google Drive remount before run", value=True)
        force_rerun_input = gr.Checkbox(label="Recompute TRIBE even if cache exists", value=False)
    with gr.Row():
        n_timesteps_input = gr.Slider(label="TRIBE timesteps to plot", minimum=1, maximum=15, value=15, step=1)
        show_stimuli_input = gr.Checkbox(label="Use official TRIBE plot with video/audio/text stimuli", value=True)

    run_button = gr.Button("Run analysis", variant="primary")

    status_output = gr.Textbox(label="Status", lines=1)
    log_output = gr.Textbox(label="Progress log", lines=22)
    paths_output = gr.Textbox(label="Output paths", lines=4)
    plot_output = gr.Plot(label="TRIBE timestep visualization")
    aggregate_output = gr.Dataframe(label="Aggregate marketing scores")
    segment_output = gr.Dataframe(label="Top-3 categories per segment")
    report_file_output = gr.File(label="Download HTML report")

    run_button.click(
        fn=analyze_video,
        inputs=[
            video_path_input,
            hf_token_input,
            root_dir_input,
            install_requirements_input,
            mount_drive_input,
            force_remount_drive_input,
            force_rerun_input,
            n_timesteps_input,
            show_stimuli_input,
        ],
        outputs=[status_output, log_output, paths_output, plot_output, aggregate_output, segment_output, report_file_output],
    )

print("Starting Gradio launch. If output pauses here, Gradio/Colab is creating the public share URL...", flush=True)
queued_demo = demo.queue()
print("Queue initialized. Calling launch(...)", flush=True)
launch_result = queued_demo.launch(
    share=True,
    debug=False,
    prevent_thread_lock=True,
    inline=False,
    quiet=False,
    show_error=True,
)

try:
    _, local_url, share_url = launch_result
except Exception:
    local_url = getattr(launch_result, "local_url", None)
    share_url = getattr(launch_result, "share_url", None)

url_text = f"local_url={local_url}\nshare_url={share_url}\n"
print("Gradio is running.", flush=True)
print(url_text, flush=True)
print("Open share_url in a browser. If the notebook output disappears, check gradio_url.txt.", flush=True)

for candidate in [Path(DEFAULT_ROOT_DIR) / "logs" / "gradio_url.txt", PROJECT_DIR / "gradio_url.txt"]:
    try:
        candidate.parent.mkdir(parents=True, exist_ok=True)
        candidate.write_text(url_text, encoding="utf-8")
        print(f"Saved Gradio URL to: {candidate}", flush=True)
        break
    except Exception as exc:
        print(f"Could not save Gradio URL to {candidate}: {exc}", flush=True)

_GRADIO_DEMO = demo
_GRADIO_LAUNCH_RESULT = launch_result
print("The cell can finish now; use the printed share_url to open the UI.", flush=True)
